In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
import psycopg2
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# Initialize Spark session
spark = SparkSession.builder.appName("F1Prediction").getOrCreate()

# Load datasets from S3
df_pitstops = spark.read.csv('s3://columbia-gr5069-main/raw/pit_stops.csv', header=True)
df_results = spark.read.csv('s3://columbia-gr5069-main/raw/results.csv', header=True)
df_drivers = spark.read.csv('s3://columbia-gr5069-main/raw/drivers.csv', header=True)
df_races = spark.read.csv('s3://columbia-gr5069-main/raw/races.csv', header=True)
df_laptimes = spark.read.csv('s3://columbia-gr5069-main/raw/lap_times.csv', header=True)
df_sprint_results = spark.read.csv('s3://columbia-gr5069-main/raw/sprint_results.csv', header=True)



In [ ]:
# Data Preprocessing
# Join results with races to get race year and circuit details
df_races_selected = df_races.select('raceId', 'year', 'circuitId')
df_results_joined = df_results.join(df_races_selected, on='raceId', how='left')

# Join with pitstops to get average pitstop duration per race
df_pitstops_avg = df_pitstops.groupBy('raceId', 'driverId').avg('milliseconds').withColumnRenamed('avg(milliseconds)', 'avg_pitstop_ms')
df_results_joined = df_results_joined.join(df_pitstops_avg, on=['raceId', 'driverId'], how='left')

# Join with laptimes to get average lap time per race
df_laptimes_avg = df_laptimes.groupBy('raceId', 'driverId').avg('milliseconds').withColumnRenamed('avg(milliseconds)', 'avg_laptime_ms')
df_results_joined = df_results_joined.join(df_laptimes_avg, on=['raceId', 'driverId'], how='left')



In [ ]:
# Select relevant features and target
df_features = df_results_joined.select(
    'driverId', 'constructorId', 'grid', 'year', 'circuitId',
    'avg_pitstop_ms', 'avg_laptime_ms', 'positionOrder'
).na.fill(0)  # Fill NA with 0 for simplicity

# Convert to Pandas for scikit-learn compatibility
pdf = df_features.toPandas()



In [ ]:
# Feature engineering
X = pdf[['grid', 'year', 'avg_pitstop_ms', 'avg_laptime_ms']].astype(float)
y = pdf['positionOrder'].astype(int)

# Split data
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



In [ ]:
# Database setup
db_params = {
    'host': 'localhost',  # Update with your DB host
    'port': '5432',      # Update with your DB port
    'database': 'f1_db', # Update with your DB name
    'user': 'your_user', # Update with your DB user
    'password': 'your_password' # Update with your DB password
}

# Create database tables
def create_tables():
    conn = psycopg2.connect(**db_params)
    cur = conn.cursor()

    # Table for Model 1 predictions
    cur.execute("""
        CREATE TABLE IF NOT EXISTS model1_predictions (
            id SERIAL PRIMARY KEY,
            race_id INTEGER,
            driver_id INTEGER,
            predicted_position INTEGER,
            actual_position INTEGER,
            prediction_timestamp TIMESTAMP
        );
    """)

    # Table for Model 2 predictions
    cur.execute("""
        CREATE TABLE IF NOT EXISTS model2_predictions (
            id SERIAL PRIMARY KEY,
            race_id INTEGER,
            driver_id INTEGER,
           动作

            predicted_position INTEGER,
            actual_position INTEGER,
            prediction_timestamp TIMESTAMP
        );
    """)

    conn.commit()
    cur.close()
    conn.close()

create_tables()

# Function to store predictions in database
def store_predictions(predictions, actuals, race_ids, driver_ids, model_name):
    conn = psycopg2.connect(**db_params)
    cur = conn.cursor()
    table_name = f"model{model_name}_predictions"
    for pred, actual, race_id, driver_id in zip(predictions, actuals, race_ids, driver_ids):
        cur.execute(f"""
            INSERT INTO {table_name} (race_id, driver_id, predicted_position, actual_position, prediction_timestamp)
            VALUES (%s, %s, %s, %s, %s)
        """, (int(race_id), int(driver_id), int(pred), int(actual), datetime.now()))
    conn.commit()
    cur.close()
    conn.close()



In [ ]:
# MLflow experiment setup
mlflow.set_experiment("F1_Position_Prediction")

# Model 1: Random Forest
with mlflow.start_run(run_name="RandomForest_Model"):
    # Hyperparameters
    rf_params = {'n_estimators': 100, 'max_depth': 10, 'random_state': 42}

    # Train model
    rf_model = RandomForestClassifier(**rf_params)
    rf_model.fit(X_train, y_train)

    # Predictions
    rf_predictions = rf_model.predict(X_test)

    # Metrics
    accuracy = accuracy_score(y_test, rf_predictions)
    precision = precision_score(y_test, rf_predictions, average='weighted')
    recall = recall_score(y_test, rf_predictions, average='weighted')
    f1 = f1_score(y_test, rf_predictions, average='weighted')

    # Log parameters and metrics
    mlflow.log_params(rf_params)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)

    # Log model
    mlflow.sklearn.log_model(rf_model, "random_forest_model")

    # Artifacts: Feature Importance Plot
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': rf_model.feature_importances_
    }).sort_values('importance', ascending=False)

    plt.figure(figsize=(8, 6))
    sns.barplot(x='importance', y='feature', data=feature_importance)
    plt.title('Random Forest Feature Importance')
    plt.savefig('rf_feature_importance.png')
    mlflow.log_artifact('rf_feature_importance.png')

    # Artifact: Confusion Matrix
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y_test, rf_predictions)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d')
    plt.title('Random Forest Confusion Matrix')
    plt.savefig('rf_confusion_matrix.png')
    mlflow.log_artifact('rf_confusion_matrix.png')

    # Store predictions
    race_ids = df_results_joined.select('raceId').toPandas()['raceId'].iloc[X_test.index]
    driver_ids = df_results_joined.select('driverId').toPandas()['driverId'].iloc[X_test.index]
    store_predictions(rf_predictions, y_test, race_ids, driver_ids, "1")



In [ ]:
# Model 2: Gradient Boosting
with mlflow.start_run(run_name="GradientBoosting_Model"):
    # Hyperparameters
    gb_params = {'n_estimators': 50, 'learning_rate': 0.1, 'max_depth': 5, 'random_state': 42}

    # Train model
    gb_model = GradientBoostingClassifier(**gb_params)
    gb_model.fit(X_train, y_train)

    # Predictions
    gb_predictions = gb_model.predict(X_test)

    # Metrics
    accuracy = accuracy_score(y_test, gb_predictions)
    precision = precision_score(y_test, gb_predictions, average='weighted')
    recall = recall_score(y_test, gb_predictions, average='weighted')
    f1 = f1_score(y_test, gb_predictions, average='weighted')

    # Log parameters and metrics
    mlflow.log_params(gb_params)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)

    # Log model
    mlflow.sklearn.log_model(gb_model, "gradient_boosting_model")

    # Artifacts: Feature Importance Plot
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': gb_model.feature_importances_
    }).sort_values('importance', ascending=False)

    plt.figure(figsize=(8, 6))
    sns.barplot(x='importance', y='feature', data=feature_importance)
    plt.title('Gradient Boosting Feature Importance')
    plt.savefig('gb_feature_importance.png')
    mlflow.log_artifact('gb_feature_importance.png')

    # Artifact: Confusion Matrix
    cm = confusion_matrix(y_test, gb_predictions)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d')
    plt.title('Gradient Boosting Confusion Matrix')
    plt.savefig('gb_confusion_matrix.png')
    mlflow.log_artifact('gb_confusion_matrix.png')

    # Store predictions
    store_predictions(gb_predictions, y_test, race_ids, driver_ids, "2")

# Stop Spark session
spark.stop()